# Typing Speeds

**Category:** Data Cleaning  
**Dataset:** typing-speeds.csv — 168,594 rows, 11 columns  
**Objective:** Determine whether taking a typing course actually improves
speed, while correctly identifying and controlling for a confounding variable.

---

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/typing-speeds.csv')
print(df.shape)
df.head(3)

(168594, 11)


,PARTICIPANT_ID,AGE,HAS_TAKEN_TYPING_COURSE,COUNTRY,LAYOUT,NATIVE_LANGUAGE,FINGERS,KEYBOARD_TYPE,ERROR_RATE,AVG_WPM_15,ROR
0,3,30,0,US,qwerty,en,1-2,full,0.511945,61.9483,0.2288
1,5,27,0,MY,qwerty,en,7-8,laptop,0.871080,72.8871,0.3675
2,7,13,0,AU,qwerty,en,7-8,laptop,6.685633,24.1809,0.0667


## 1. Dataset Exploration

This is by far the largest dataset in the portfolio. Running the standard
exploration sequence, paying special attention to AGE, since self-reported
survey data commonly contains implausible values.

In [2]:
print("Missing values per column: ")
print(df.isnull().sum())
print()
print("AGE summary statistics: ")
print(df['AGE'].describe())

Missing values per column: 
PARTICIPANT_ID              0
AGE                         0
HAS_TAKEN_TYPING_COURSE     0
COUNTRY                    20
LAYOUT                      0
NATIVE_LANGUAGE             0
FINGERS                     0
KEYBOARD_TYPE               0
ERROR_RATE                  0
AVG_WPM_15                  0
ROR                         0
dtype: int64

AGE summary statistics: 
count    168594.000000
mean         24.571859
std          11.239181
min           0.000000
25%          17.000000
50%          22.000000
75%          29.000000
max         120.000000
Name: AGE, dtype: float64


In [3]:
print("Rows with Age == 0: ", (df['AGE']==0).sum())
print("Rows with Age < 5: ", (df['AGE'] < 5).sum())
print("Rows with AGE > 95: ", (df['AGE'] > 95).sum())

Rows with Age == 0:  181
Rows with Age < 5:  1147
Rows with AGE > 95:  382


## 2. Cleaning: Filter Implausible Ages

The `AGE` column contains values from **0** to **120**, which include implausible ages for independent participants. A range of **5–95 years** was selected as a reasonable threshold before analysis, and the percentage of removed rows is reported to document the impact of this cleaning step.

In [ ]:
before_count = len(df)

df_clean = df[(df['AGE'] >= 5) & (df['AGE'] <= 95)].copy()

after_count = len(df_clean)

removed = before_count - after_count
pct_removed = removed / before_count * 100

print(f"Removed {removed} rows ({pct_removed:.2f}%) with implausible ages.")
print(f"Remaning Rows: {after_count}")

Removed 1529 rows (0.91%) with implausible ages.
Remaning Rows: 167065


## 3. Cleaning: Handle Missing Country

Only **20 of 168,594** records contain a missing `COUNTRY` value (approximately **0.01%** of the dataset). Since this proportion is negligible and `COUNTRY` is not central to the analysis, these records can be removed.

In [8]:
print("Missing country values:", df_clean['COUNTRY'].isnull().sum())

df_clean = df_clean.dropna(subset='COUNTRY')

print("Rows after dropping missing countries:", len(df_clean))

Missing country values: 0
Rows after dropping missing countries: 167045


## 4. Naive Comparison: Course vs. No Course

A simple comparison of average typing speed between participants who have and have not taken a typing course provides an initial view of the data. However, this comparison does not account for other factors that may also influence typing speed.

In [9]:
naive_comparision = df_clean.groupby('HAS_TAKEN_TYPING_COURSE')['AVG_WPM_15'].agg(['mean', 'count'])
naive_comparision

,mean,count
HAS_TAKEN_TYPING_COURSE,,
0,49.019926,115522
1,54.383313,51523


## 6. Confound-Controlled Comparison

Typing speed is compared by course status **within each finger-count group** rather than across the entire dataset. This controls for differences in the number of fingers used, allowing for a fairer comparison of participants with similar typing techniques.

In [13]:
controlled_comparison = df_clean.groupby(['FINGERS', 'HAS_TAKEN_TYPING_COURSE'])['AVG_WPM_15'].mean().unstack()
controlled_comparison['gap'] = controlled_comparison[1] - controlled_comparison[0]
controlled_comparison.round(2)

HAS_TAKEN_TYPING_COURSE,0,1,gap
FINGERS,,,
1-2,40.04,39.23,-0.81
10+,42.79,NaN,NaN
3-4,41.11,40.63,-0.48
5-6,46.00,44.61,-1.39
7-8,50.13,49.91,-0.22
9-10,56.06,59.24,3.17
